# HDBSCAN Battery: Performance on Agent-Generated Trajectories

Evaluate `st_hdbscan` across a battery of realistic synthetic trajectories produced by
the `Agent` simulation (EPR mobility model on the Garden City map). Ground truth comes
directly from `agent.diary`.

**Sweep dimensions**

| Dimension | Values | What it controls |
|---|---|---|
| `seed` | 0–9 | Agent home/workplace and movement |
| `beta_start` | 100, 250, 400 | Burst inter-arrival time → trajectory sparsity |
| `time_thresh` | 20, 60 | HDBSCAN temporal neighborhood (min) |
| `min_pts` | 2, 3 | HDBSCAN core-point threshold |

**Metrics** are time-based (seconds of overlap between detected and ground-truth stop intervals):
- **Precision** — detected stop time that overlaps a true stop / total detected stop time  
- **Recall** — true stop time covered by any detected stop / total true stop time  
- **F1** — harmonic mean

In [ ]:
%matplotlib inline
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nomad.city_gen import City
from nomad.traj_gen import Agent
import nomad.stop_detection.hdbscan as HDBSCAN
import nomad.data as data_folder

pd.set_option('display.float_format', '{:.3f}'.format)
print('imports OK')

## 1. Load

Building hub network, gravity matrix, and shortest paths are computed once and reused for all agents.

In [ ]:
data_dir = Path(data_folder.__file__).parent
city_path = data_dir / 'garden-city.gpkg'

t0 = time.perf_counter()
city = City.from_geopackage(str(city_path))
city._build_hub_network(hub_size=16)
city.compute_gravity(exponent=2.0)
city.compute_shortest_paths(callable_only=True)
print(f'City loaded in {time.perf_counter() - t0:.1f}s')
print(f'  Buildings: {len(city.buildings_gdf)}')
print(f'  Dimensions: {city.dimensions}')

# Pre-sample a pool of home/workplace pairs for the agents
rng_pool = np.random.default_rng(0)
homes      = city.buildings_gdf[city.buildings_gdf['building_type'] == 'home']['id'].to_numpy()
workplaces = city.buildings_gdf[city.buildings_gdf['building_type'] == 'workplace']['id'].to_numpy()
N_AGENTS = 10
home_pool = rng_pool.choice(homes,      size=N_AGENTS, replace=True)
work_pool = rng_pool.choice(workplaces, size=N_AGENTS, replace=True)

TC = {'timestamp': 'timestamp', 'x': 'x', 'y': 'y'}

## 2. Metrics

In [ ]:
def _overlap(a_s, a_e, b_s, b_e):
    return max(0, min(a_e, b_e) - max(a_s, b_s))


def temporal_metrics(stops_df, gt, tc):
    """
    Time-based precision / recall / F1.

    Parameters
    ----------
    stops_df : output of st_hdbscan with complete_output=True
    gt       : DataFrame with columns start_ts (int, seconds) and end_ts (int, seconds)
    tc       : traj_cols dict (must include 'timestamp')
    """
    if stops_df is None or stops_df.empty:
        return dict(precision=0.0, recall=0.0, f1=0.0, n_detected=0)

    detected = [
        (int(r[tc['timestamp']]), int(r['end_timestamp']))
        for _, r in stops_df.iterrows()
    ]
    truth = [
        (int(r['start_ts']), int(r['end_ts']))
        for _, r in gt.iterrows()
    ]

    total_det = sum(e - s for s, e in detected)
    total_gt  = sum(e - s for s, e in truth)

    tp = sum(_overlap(ds, de, gs, ge) for ds, de in detected for gs, ge in truth)

    precision = tp / total_det if total_det > 0 else 0.0
    recall    = tp / total_gt  if total_gt  > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )
    return dict(
        precision=round(precision, 4),
        recall=round(recall, 4),
        f1=round(f1, 4),
        n_detected=len(stops_df),
    )


def diary_to_gt(diary):
    """
    Convert agent.diary to a ground-truth stop interval DataFrame.
    Skips transit entries (location is None) and consecutive duplicate
    locations are merged so back-to-back entries at the same building
    count as one stop.
    """
    df = diary[diary['location'].notna()].copy()
    df['end_ts'] = df['timestamp'] + (df['duration'] * 60).astype(int)
    df = df.rename(columns={'timestamp': 'start_ts'})

    # Merge consecutive entries at the same building
    rows = []
    for _, r in df.iterrows():
        if rows and rows[-1]['location'] == r['location'] and r['start_ts'] <= rows[-1]['end_ts']:
            rows[-1]['end_ts'] = r['end_ts']
        else:
            rows.append({'start_ts': r['start_ts'], 'end_ts': r['end_ts'],
                         'location': r['location']})

    gt = pd.DataFrame(rows)
    gt['duration_min'] = (gt['end_ts'] - gt['start_ts']) / 60
    return gt

## 3. Battery Configuration

In [ ]:
# Trajectory generation window
START_DT  = pd.Timestamp('2024-01-01T07:00-04:00')
END_DT    = pd.Timestamp('2024-01-02T07:00-04:00')  # 1-day trajectories

# Sampling sparsity: higher beta_start → more time between bursts → sparser
BETA_START_VALS = [100, 250, 400]   # burst inter-arrival time (min)
BETA_PING       = 5                 # ping inter-arrival within burst (min)

# HDBSCAN configurations
HDBSCAN_CONFIGS = [
    dict(time_thresh=20, min_pts=2, min_cluster_size=2, dur_min=5,  label='tt=20 mp=2 mcs=2'),
    dict(time_thresh=60, min_pts=2, min_cluster_size=2, dur_min=5,  label='tt=60 mp=2 mcs=2'),
    dict(time_thresh=60, min_pts=3, min_cluster_size=2, dur_min=5,  label='tt=60 mp=3 mcs=2'),
    dict(time_thresh=60, min_pts=2, min_cluster_size=1, dur_min=5,  label='tt=60 mp=2 mcs=1'),
]

total_runs = N_AGENTS * len(BETA_START_VALS) * len(HDBSCAN_CONFIGS)
print(f'Agents: {N_AGENTS}  |  Sparsity configs: {len(BETA_START_VALS)}  |  HDBSCAN configs: {len(HDBSCAN_CONFIGS)}')
print(f'Total HDBSCAN runs: {total_runs}')

## 4. Generate Trajectories

In [ ]:
# Generate dense ground-truth trajectories once per agent/seed
agents = {}
for seed in range(N_AGENTS):
    agent = Agent(
        identifier=f'agent_{seed:03d}',
        city=city,
        home=home_pool[seed],
        workplace=work_pool[seed],
        datetime=START_DT,
        seed=seed,
    )
    agent.generate_trajectory(end_time=END_DT, seed=seed)
    agents[seed] = agent

print(f'Generated {len(agents)} dense trajectories')
# Peek at one diary
ex = agents[0]
print(f'\nAgent 0 diary ({len(ex.diary)} entries, {len(ex.trajectory)} ticks):')
ex.diary.head(8)

## 5. Run Battery

In [ ]:
records = []

for seed, agent in agents.items():
    gt = diary_to_gt(agent.diary)

    for beta_start in BETA_START_VALS:
        # Sample sparse observations with this sparsity setting
        agent.sample_trajectory(
            beta_ping=BETA_PING,
            beta_start=beta_start,
            beta_durations=beta_start,  # burst duration ~ inter-arrival
            replace_sparse_traj=True,
            seed=seed,
        )
        sparse = agent.sparse_traj
        n_pings = len(sparse)
        q = n_pings / len(agent.trajectory)  # completeness

        for cfg in HDBSCAN_CONFIGS:
            params = {k: v for k, v in cfg.items() if k != 'label'}

            t0 = time.perf_counter()
            try:
                stops = HDBSCAN.st_hdbscan(
                    sparse.copy(),
                    traj_cols=TC,
                    complete_output=True,
                    **params,
                )
            except Exception as exc:
                print(f'  seed={seed} beta_start={beta_start} {cfg["label"]}: {exc}')
                stops = None
            elapsed = time.perf_counter() - t0

            m = temporal_metrics(stops, gt, TC)
            records.append({
                'seed':         seed,
                'beta_start':   beta_start,
                'q':            round(q, 3),
                'n_pings':      n_pings,
                'n_true_stops': len(gt),
                'hdbscan_cfg':  cfg['label'],
                'runtime_s':    round(elapsed, 4),
                **m,
            })

raw_df = pd.DataFrame(records)
print(f'Done. {len(raw_df)} rows.')
raw_df.head(8)

## 6. Summary Tables

### 6a. Overall: mean ± std per HDBSCAN config (across agents and sparsities)

In [ ]:
summary = (
    raw_df
    .groupby('hdbscan_cfg')[['precision', 'recall', 'f1', 'runtime_s']]
    .agg(['mean', 'std'])
    .round(3)
)
summary.columns = [f'{col}_{stat}' for col, stat in summary.columns]
summary.reset_index()

### 6b. Pivot: HDBSCAN config × trajectory sparsity → mean F1

In [ ]:
pivot = (
    raw_df
    .groupby(['hdbscan_cfg', 'beta_start'])['f1']
    .mean()
    .round(3)
    .unstack('beta_start')
)
pivot.columns.name = 'beta_start (sparsity →)'
pivot

### 6c. Full aggregate: config × beta_start × seed (mean over HDBSCAN configs)

In [ ]:
agg = (
    raw_df
    .groupby(['hdbscan_cfg', 'beta_start', 'seed'])
    [['q', 'n_pings', 'n_true_stops', 'precision', 'recall', 'f1', 'n_detected', 'runtime_s']]
    .first()  # one row per (cfg, beta_start, seed) combination
    .reset_index()
)
agg

## 7. Visualisations

### 7a. F1 vs trajectory completeness (q), by HDBSCAN config

In [ ]:
configs  = [cfg['label'] for cfg in HDBSCAN_CONFIGS]
palette  = dict(zip(configs, sns.color_palette('tab10', n_colors=len(configs))))

fig, ax = plt.subplots(figsize=(7, 4))
for cfg_label in configs:
    sub = raw_df[raw_df['hdbscan_cfg'] == cfg_label].sort_values('q')
    ax.scatter(sub['q'], sub['f1'], s=20, alpha=0.5, color=palette[cfg_label])
    grouped = sub.groupby('beta_start')['f1'].mean()
    q_means = sub.groupby('beta_start')['q'].mean()
    ax.plot(q_means.values, grouped.values, marker='o', lw=2,
            color=palette[cfg_label], label=cfg_label)

ax.set_xlabel('q  (trajectory completeness)')
ax.set_ylabel('F1')
ax.set_ylim(0, 1.05)
ax.legend(title='HDBSCAN config', fontsize=9)
ax.set_title('F1 vs trajectory completeness')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7b. Precision / Recall / F1 bar chart by HDBSCAN config

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, metric in zip(axes, ['precision', 'recall', 'f1']):
    means = raw_df.groupby('hdbscan_cfg')[metric].mean().reindex(configs)
    stds  = raw_df.groupby('hdbscan_cfg')[metric].std().reindex(configs)
    bars  = ax.bar(range(len(configs)), means.values, yerr=stds.values,
                   color=[palette[c] for c in configs], capsize=4, alpha=0.85)
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels(configs, rotation=25, ha='right', fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_title(metric.capitalize())
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Mean ± std across agents and sparsity levels')
plt.tight_layout()
plt.show()

### 7c. F1 heatmap: agent seed × sparsity (beta_start), per HDBSCAN config

In [ ]:
fig, axes = plt.subplots(1, len(configs), figsize=(5 * len(configs), 4), sharey=True)

for ax, cfg_label in zip(axes, configs):
    pivot = (
        raw_df[raw_df['hdbscan_cfg'] == cfg_label]
        .groupby(['seed', 'beta_start'])['f1']
        .mean()
        .unstack('beta_start')
    )
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt='.2f',
        vmin=0, vmax=1, cmap='YlGnBu',
        cbar=(ax is axes[-1]),
    )
    ax.set_title(cfg_label, fontsize=10)
    ax.set_xlabel('beta_start (sparsity)')
    ax.set_ylabel('agent seed' if ax is axes[0] else '')

fig.suptitle('F1 score per agent × sparsity', y=1.02)
plt.tight_layout()
plt.show()

### 7d. Sample trajectory: map + stop timeline for one agent

In [ ]:
DEMO_SEED       = 0
DEMO_BETA_START = 250

agent = agents[DEMO_SEED]
gt    = diary_to_gt(agent.diary)
agent.sample_trajectory(
    beta_ping=BETA_PING,
    beta_start=DEMO_BETA_START,
    beta_durations=DEMO_BETA_START,
    replace_sparse_traj=True,
    seed=DEMO_SEED,
)
sparse = agent.sparse_traj

# Pick best config by overall mean F1
best_cfg_label  = raw_df.groupby('hdbscan_cfg')['f1'].mean().idxmax()
best_cfg_params = {k: v for k, v in next(c for c in HDBSCAN_CONFIGS if c['label'] == best_cfg_label).items() if k != 'label'}
stops = HDBSCAN.st_hdbscan(sparse.copy(), traj_cols=TC, complete_output=True, **best_cfg_params)
m = temporal_metrics(stops, gt, TC)
print(f'Config: {best_cfg_label}  |  P={m["precision"]:.2f}  R={m["recall"]:.2f}  F1={m["f1"]:.2f}')

fig, (ax_map, ax_time) = plt.subplots(
    2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [8, 1]}
)

# Map
city.plot_city(ax=ax_map, doors=False, address=False)
ax_map.scatter(sparse['x'], sparse['y'], s=10, color='black', alpha=0.5, zorder=3, label='pings')
cmap = plt.get_cmap('tab10')
for j, (_, srow) in enumerate(stops.iterrows()):
    circle = plt.Circle(
        (srow['x'], srow['y']),
        radius=max(15, srow.get('diameter', 30) / 2),
        color=cmap(j % 10), alpha=0.35, zorder=2,
    )
    ax_map.add_patch(circle)
    ax_map.scatter(srow['x'], srow['y'], s=80, color=cmap(j % 10), zorder=4)
ax_map.set_axis_off()
ax_map.set_title(
    f'Agent {DEMO_SEED} | beta_start={DEMO_BETA_START} | {len(sparse)} pings\n'
    f'{best_cfg_label} → P={m["precision"]:.2f}  R={m["recall"]:.2f}  F1={m["f1"]:.2f}'
)

# Timeline
t_min = int(sparse['timestamp'].min())
t_max = int(sparse['timestamp'].max())
for k, gtr in gt.iterrows():
    ax_time.barh(0, gtr['end_ts'] - gtr['start_ts'], left=gtr['start_ts'] - t_min,
                 height=0.4, color='steelblue', alpha=0.7,
                 label='ground truth' if k == 0 else None)
for j, (_, srow) in enumerate(stops.iterrows()):
    s_start = int(srow[TC['timestamp']]) - t_min
    s_end   = int(srow['end_timestamp']) - t_min
    ax_time.barh(-0.5, s_end - s_start, left=s_start, height=0.4,
                 color=cmap(j % 10), alpha=0.9,
                 label='detected' if j == 0 else None)
ax_time.set_xlim(0, t_max - t_min)
ax_time.set_yticks([0, -0.5])
ax_time.set_yticklabels(['truth', 'detected'])
ax_time.set_xlabel('seconds elapsed')
ax_time.legend(fontsize=9)
ax_time.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Export

In [ ]:
# Uncomment to save
# raw_df.to_csv('hdbscan_battery_raw.csv', index=False)
print('Raw rows:', len(raw_df))
print('Unique agents:', raw_df['seed'].nunique())
print('Unique configs:', raw_df['hdbscan_cfg'].nunique())
raw_df.groupby(['hdbscan_cfg', 'beta_start'])[['precision','recall','f1']].mean().round(3)